1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 답변 생성이 오래걸림
3. 임베딩 --> 벡터 데이터 베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

python.langchain 들어가면 docs 변화 방법 있음

In [57]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax_with_markdown.docx')
document_list = loader.load_and_split(text_splitter)

In [58]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

In [ ]:
from langchain_pinecone import PineconeVectorStore

index_name ='tax-with-markdown' #'tax-table-index' #'tax-upstage-index'

database = PineconeVectorStore.from_existing_index(embedding = embedding, index_name = index_name)

In [60]:
import os

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

import time

from pinecone import Pinecone 

pc = Pinecone(api_key=pinecone_api_key)

In [102]:
query ='연봉 5천만원인 직장인의 종합소득세는 얼마인가요?'
#retrieved_docs = database.similarity_search(query, k=3)

In [103]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

In [104]:
# prompt = f"""[Identity]
# - 당신은 최고의 한국 소득세 전문가 입니다
# - [Context]를 참고해서 사용자의 질문에 답변해주세요

# [Context]
# {retrieved_docs}

# Question: {query}
# """

In [105]:
# ai_message = llm.invoke(prompt)

In [106]:
# ai_message.content

In [107]:
from langchain_classic import hub

prompt = hub.pull("rlm/rag-prompt")

In [108]:
# from langchain_classic.chains import RetrievalQA

# qa_chain = RetrievalQA.from_chain_type(
#     llm,
#     retriever=database.as_retriever(), 
#     chain_type_kwargs={"prompt":prompt},
# )

In [109]:
# retriveal 효율 개선

from langchain_classic.chains import RetrievalQA

retriever=database.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever, 
    chain_type_kwargs={"prompt":prompt},
)

In [110]:
retriever.invoke(query)

[Document(id='4c4a939d-e34d-467d-858a-6bb97d2b0931', metadata={'source': './tax_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n\n\n\n\n② 거주자의 퇴직소득에 대한 소

In [111]:
ai_message = qa_chain({"query":query})

In [112]:
ai_message

{'query': '연봉 5천만원인 직장인의 종합소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 종합소득세는 400만원입니다. 이는 과세표준 5,000만원 이하 구간의 세율에 해당하는 금액입니다.'}

In [113]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고, 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    사전: {dictionary}
    """)

dictionary_chain = prompt | llm | StrOutputParser()

In [114]:
new_question = dictionary_chain.invoke({"question": query})

In [115]:
query

'연봉 5천만원인 직장인의 종합소득세는 얼마인가요?'

In [116]:
new_question

'사용자의 질문에 \'사람을 나타내는 표현\'이 포함되어 있다면, 이를 \'거주자\'로 변경하여 질문을 수정합니다. 만약 그러한 표현이 없다면, 질문을 그대로 유지합니다.\n\n예를 들어, 사용자의 원래 질문이 "사람들이 왜 도시로 이동하는 걸까요?"라면, 변경 후의 질문은 "거주자들이 왜 도시로 이동하는 걸까요?"가 됩니다.\n\n변경해야 할 내용이 없다면, 질문은 그대로 유지됩니다. 구체적인 질문을 제시해 주시면 더 정확한 변경을 도와드릴 수 있습니다.'

'사용자의 질문에 \'사람을 나타내는 표현\'이 포함되어 있다면, 이를 \'거주자\'로 변경하여 질문을 수정합니다. 만약 질문에서 해당 표현이 발견되지 않는다면, 질문을 변경하지 않습니다.\n\n예를 들어:\n- 원래 질문: "사람들은 왜 도시에서 살기를 선호하나요?"\n- 수정된 질문: "거주자들은 왜 도시에서 살기를 선호하나요?"\n\n질문을 보내주시면, 사전을 참고하여 필요한 경우 변경해드리겠습니다.'

In [117]:
tax_chain = {"query": dictionary_chain} | qa_chain

In [118]:
ai_response = tax_chain.invoke({"question":query})

In [119]:
ai_response

{'query': '사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해드리겠습니다. 만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않을 것입니다.\n\n사전: [\'사람을 나타내는 표현 -> 거주자\']\n\n예시:\n사용자 질문: "사람이 새로 개발된 도시로 이주하는 이유는 무엇인가요?"\n\n변경된 질문: "거주자가 새로 개발된 도시로 이주하는 이유는 무엇인가요?"\n\n만약 사용자의 질문이 사전의 어떤 표현과도 관련이 없다면, 질문은 그대로 유지됩니다. 구체적인 질문을 주시면, 그에 맞게 변경해드리겠습니다.',
 'result': '사용자의 질문이 제공된 사전의 표현과 관련이 없으므로, 질문을 변경하지 않습니다.'}